In [33]:
import numpy as np
import matplotlib.pyplot as plt

m,cm=1,0.01
class Detector():
    def __init__(self, R=2*m,N=40,r=10*cm, prob=0.5):
        self.R=R
        self.N=N
        self.r=r
        self.prob=prob
    def random_distribution(self):
        phi=np.random.uniform(0,2*np.pi, self.N)
        cosTheta=np.random.uniform(-1,1,self.N)
        sinTheta=np.sqrt(1-cosTheta*cosTheta)
        return self.R*np.array([sinTheta*np.cos(phi),sinTheta*np.sin(phi), cosTheta])
    #здесь метод проверки попадания в ФЭУ изменен в отличие от задания 1 под задание 4 (учет сдвига источника из центра). 
    def IsInPMT(self,origin:np.ndarray,PhotonArray:np.ndarray, PMTarray:np.ndarray):
        cosMax=np.cos(self.r/self.R)
        cosCheck=(origin[:, None]+PhotonArray).T@PMTarray
        return np.any(cosCheck>(self.R**2)*cosMax, axis=1)
        
    def ElectronYield(self, NDetected:float):
        return np.random.poisson(self.prob * NDetected) #по свойству распределения пуассона 



class Source():
    def __init__(self, E:np.ndarray,origin:np.ndarray,Y=1e4):
        self.Y=Y
        self.N=int(E*Y)
        self.origin=origin
    def random_distribution(self):
        phi=np.random.uniform(0,2*np.pi, self.N)
        cosTheta=np.random.uniform(-1,1,self.N)
        sinTheta=np.sqrt(1-cosTheta*cosTheta)
        return np.array([sinTheta*np.cos(phi),sinTheta*np.sin(phi), cosTheta])
    def distance_traversed(self,PhotonArray:np.ndarray,R:float):
        self.origin=np.reshape(self.origin,(1,3))
        t=-self.origin@PhotonArray+np.sqrt(R**2-np.sum(self.origin**2)+(self.origin@PhotonArray)*(self.origin@PhotonArray))
        return t

D1=Detector()
PMTarray=D1.random_distribution()

origin=np.array([0,0,1])
S1=Source(1,origin)

PhotonArray=S1.random_distribution()
#в случае origin=0 distance_traversed=D1.R для каждого фотона
PhotonArray*=S1.distance_traversed(PhotonArray,D1.R)

registered=np.sum(D1.IsInPMT(origin,PhotonArray,PMTarray))
                
print (f"electron count: {D1.ElectronYield(registered)}, predicted: {int(D1.prob*S1.N*D1.N*D1.r**2/4/D1.R**2)}")
 
    
    
          


electron count: 118, predicted: 125


Так как количество выбитых фотонов в ФЭУ является чисто характеристикой самого ФЭУ, метод, реализующий пуссоновское распределение фотоэлектронов ElectronYield, был реализован в классе Detector. Число образованных фотоэлектронов флуктуирует вокруг ожидаемого. 